# DocuMind — RAG Pipeline Walkthrough

Sections:
1. Load & chunk a document
2. Embed chunks locally (no API key needed)
3. Store & query the vector index
4. Inspect retrieval quality
5. Generate a grounded answer with Gemini
6. A small precision@k evaluation

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from app.ingest import load_text, chunk_text
from app import vectorstore, llm, rag
from app.config import settings

print("Embedding model:", settings.embedding_model)
print("Generation model:", settings.gemini_model)
print("Chunk size / overlap:", settings.chunk_size, "/", settings.chunk_overlap)

Embedding model: all-MiniLM-L6-v2
Generation model: gemini-3.1-flash-lite
Chunk size / overlap: 800 / 120


## 1. Load & chunk a document

Chunking matters alot. Chunk too large and irrelevant text dilutes
the embedding, hurting retrieval precision. Chunk too small and you lose context the
LLM needs to answer well. This project chunks on paragraph boundaries up to a target
size, then stitches a small overlap between neighbors so an answer that straddles a
chunk boundary doesn't get cut in half.

In [2]:
text = load_text("../data/sample_docs/sample.txt")
chunks = chunk_text(text)

print(f"Document length: {len(text)} chars -> {len(chunks)} chunks\n")
for i, c in enumerate(chunks):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c[:200], "...\n")

Document length: 1776 chars -> 4 chunks

--- chunk 0 (283 chars) ---
DocuMind Project Overview
DocuMind is a Retrieval-Augmented Generation (RAG) system. It lets a user upload
their own documents and ask natural-language questions about them, receiving
answers that are ...

--- chunk 1 (662 chars) ---
bout them, receiving
answers that are grounded in the actual document content rather than the
model's general knowledge. How it works:
1. Documents are split into overlapping text chunks.
2. Each chun ...

--- chunk 2 (547 chars) ---
most relevant passages are inserted into a prompt sent to Claude, which
   generates a final answer citing its sources. Why this architecture matters for engineers:
RAG systems solve the problem of la ...

--- chunk 3 (638 chars) ---
production systems such as
customer support bots, internal knowledge bases, legal research tools, and
coding assistants. Key components an AI engineer should understand:
- Chunking strategy: how docum ...



## 2. Embed chunks locally

Embeddings turn text into vectors where semantic closeness becomes geometric closeness.
We use `sentence-transformers/all-MiniLM-L6-v2` — small, fast, free, runs on CPU

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(settings.embedding_model)
vecs = model.encode(chunks[:2])
print("Embedding shape per chunk:", vecs.shape)
print("First 8 dims of chunk 0:", vecs[0][:8])

C:\Users\lenovo\AI_Portfolio\rag_qa_documind\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Embedding shape per chunk: (2, 384)
First 8 dims of chunk 0: [-0.0703031   0.06364276  0.01376749  0.02079637  0.01722235  0.0094023
 -0.04487496  0.02394657]


## 3. Store & query the vector index

This uses the same `vectorstore.py` module the app calls.

In [4]:
from app.ingest import ingest_directory

vectorstore.reset_collection()
summary = ingest_directory("../data/sample_docs")
print(summary)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


{'sample.txt': 4}


## 4. Inspect retrieval quality

Before ever calling an LLM, it's worth looking at *what gets retrieved* for a query.
If the top-k chunks aren't actually relevant, no amount of prompt engineering will
fix the final answer — this is the most common failure point in real RAG systems.

In [5]:
question = "How does DocuMind decide which text is relevant to a question?"
hits = vectorstore.query(question, top_k=3)

for h in hits:
    print(f"score={h['score']:.3f}  source={h['source']}")
    print(h['text'][:200], "...\n")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


score=0.594  source=sample.txt
DocuMind Project Overview
DocuMind is a Retrieval-Augmented Generation (RAG) system. It lets a user upload
their own documents and ask natural-language questions about them, receiving
answers that are ...

score=0.448  source=sample.txt
bout them, receiving
answers that are grounded in the actual document content rather than the
model's general knowledge. How it works:
1. Documents are split into overlapping text chunks.
2. Each chun ...

score=0.335  source=sample.txt
production systems such as
customer support bots, internal knowledge bases, legal research tools, and
coding assistants. Key components an AI engineer should understand:
- Chunking strategy: how docum ...



## 5. Generate a grounded answer

The retrieved chunks are inserted into a system-prompted call to Gemini (via
the Interactions API) that's instructed to answer *only* from context and to
cite sources — this is what keeps RAG answers from hallucinating facts not
present in your documents.



In [6]:
result = rag.answer_question(question)
print("ANSWER:\n", result["answer"])
print("\nSOURCES:", result["sources"])

ANSWER:
 DocuMind decides which text is relevant by converting the user's question into a vector embedding and comparing it against stored document chunks using cosine similarity to identify the most relevant passages.

Sources: sample.txt

SOURCES: [{'source': 'sample.txt', 'score': 0.594, 'text': "DocuMind Project Overview\nDocuMind is a Retrieval-Augmented Generation (RAG) system. It lets a user upload\ntheir own documents and ask natural-language questions about them, receiving\nanswers that are grounded in the actual document content rather than the\nmodel's general knowledge."}, {'source': 'sample.txt', 'score': 0.448, 'text': "bout them, receiving\nanswers that are grounded in the actual document content rather than the\nmodel's general knowledge. How it works:\n1. Documents are split into overlapping text chunks.\n2. Each chunk is converted into a vector embedding using a local\n   sentence-transformers model (no external API call required).\n3. Embeddings are stored in a Chrom

## 6. A minimal precision@k evaluation

A real evaluation harness is what separates a demo from a project , Here's a tiny example: for a handful of question/expected-source pairs,
check whether the correct source shows up in the top-k retrieved chunks.


In [7]:
eval_set = [
    {"question": "What embedding model does DocuMind use?", "expected_source": "sample.txt"},
    {"question": "Why use a vector database instead of retraining a model?", "expected_source": "sample.txt"},
]

correct = 0
for case in eval_set:
    hits = vectorstore.query(case["question"], top_k=3)
    retrieved_sources = {h["source"] for h in hits}
    hit = case["expected_source"] in retrieved_sources
    correct += hit
    print(f"[{'HIT' if hit else 'MISS'}] {case['question']}")

print(f"\nPrecision@3 (source-level): {correct}/{len(eval_set)}")

[HIT] What embedding model does DocuMind use?
[HIT] Why use a vector database instead of retraining a model?

Precision@3 (source-level): 2/2


## Takeaways

- Retrieval quality is the bottleneck, not the LLM. Inspect it directly before blaming generation.
- Grounding the prompt with explicit "answer only from context" instructions materially reduces hallucination.
- PDF text extraction quality matters more than people expect — some PDFs (especially LaTeX-generated academic PDFs) extract with missing spaces between words, silently degrading retrieval. Worth inspecting raw extracted text, not just final answers, when debugging a RAG pipeline.
- A tiny eval harness like the one above is easy to extend into a real regression test suite — worth building out further before calling a RAG project "done".